# 🏛️ NovusMundus · Cuaderno Maestro de Arquitectura Financiera & Trading Cuantitativo
> **Diseñado para Gemini Notebooks & Google NotebookLM**
> Autor: Mikel Irastorza · Presidente Supremo NovusMundus (Nivel 0)

Este cuaderno interactivo contiene las deducciones formales, invariantes matemáticas y código ejecutable de la plataforma de trading y contabilidad por partida doble.

## 1. Invariante Formal de Partida Doble (Pacioli 1494)
En cualquier sistema de ledger financiero de grado institucional, se satisface la ecuación fundamental:

$$\sum_{i=1}^{n} \text{Debe}_i \equiv \sum_{i=1}^{n} \text{Haber}_i$$

Para prevenir errores de redondeo de punto flotante binario (IEEE 754), todos los importes se representan estrictamente como enteros `BIGINT` en unidades mínimas indivisibles (céntimos de euro o satoshis).

In [ ]:
def verify_double_entry(entries: list[dict[str, int]]) -> bool:
    total_debit = sum(e['cents'] for e in entries if e['direction'] == 'DEBIT')
    total_credit = sum(e['cents'] for e in entries if e['direction'] == 'CREDIT')
    delta = total_debit - total_credit
    print(f'Total Débitos: {total_debit} céntimos | Total Créditos: {total_credit} céntimos')
    assert delta == 0, f'Descuadre contable detectado: {delta} céntimos'
    return True

# Prueba de libro diario
test_transfer = [
    {'account': 'ACC-TREASURY', 'direction': 'DEBIT', 'cents': 25000},
    {'account': 'ACC-SETTLEMENT', 'direction': 'CREDIT', 'cents': 25000}
]
print('Verificación Pacioli:', verify_double_entry(test_transfer))

## 2. Detección Antifraude mediante Z-Score Robusto MAD
Frente a la media y desviación estándar tradicionales que se contaminan ante outliers extremos (breakdown point 0%), la Mediana de las Desviaciones Absolutas respecto a la Mediana (**MAD**) posee un breakdown point del 50%:

$$\text{MAD} = \text{mediana}(|X_i - \tilde{X}|)$$
$$Z_{\text{robusto}} = \frac{X_i - \tilde{X}}{1.4826 \cdot \text{MAD}}$$

In [ ]:
import numpy as np

def robust_mad_z_score(data: list[float], value: float) -> float:
    arr = np.array(data, dtype=np.float64)
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    scale = 1.4826 * mad
    if scale < 1e-9:
        return 0.0
    return float((value - med) / scale)

stream = [10.2, 11.5, 9.8, 12.1, 10.5, 11.0, 950.0]  # 950.0 es un outlier de fraude
print('Z-Score Robusto para 950.0 €:', round(robust_mad_z_score(stream, 950.0), 2))

## 3. Dimensionamiento Determinista de Colas (Ley de Little)
Para evitar desbordamientos de memoria en el pipeline de streaming de 35.000 msg/s, la utilización $\rho$ de los nodos computacionales debe respetar la cota asintótica:

$$\rho = \frac{\lambda}{\mu} \le 0.80$$

Donde $\lambda$ es la tasa de llegada media y $\mu$ es la tasa de servicio por ejecutor. Cuando $\rho > 0.80$, la longitud media de la cola $L_q = \frac{\rho^2}{1-\rho}$ diverge cuadráticamente hacia el infinito.

In [ ]:
def verify_littles_law_bound(lam: float, mu: float) -> dict[str, float | bool]:
    rho = lam / max(0.001, mu)
    l_q = (rho ** 2) / max(0.001, (1.0 - rho)) if rho < 1.0 else float('inf')
    compliant = rho <= 0.80
    print(f'Tasa llegada λ: {lam:.1f} msg/s | Capacidad μ: {mu:.1f} msg/s | Utilización ρ: {rho:.3f}')
    print(f'Cola estimada Lq: {l_q:.2f} items | Criterio Little: {"ÓPTIMO ✅" if compliant else "ALERTA SATURACIÓN ⚠️"}')
    return {'rho': round(rho, 3), 'l_q': round(l_q, 2), 'compliant': compliant}

print('Prueba Ley de Little (35k msg/s / 45k capacidad):', verify_littles_law_bound(35000.0, 45000.0))

## 4. Auditoría de Ciberseguridad Zero-Trust & CISO Aegis (100/100 A+)
La defensa en profundidad implementa:
1. **AST White-Listing**: Solo se permiten sentencias `SELECT` y `WITH ... SELECT` de una única sentencia. Sentencias destructivas (`DROP`, `DELETE`, `ATTACH`, `PRAGMA query_only=OFF`) son rechazadas antes de llegar a la base de datos.
2. **Loopback Mutation Guard**: Endpoints modificadores restringidos a loopback local (`127.0.0.1`, `::1`).
3. **Hash Chain SHA-256**: Sellos criptográficos inmutables en cada transacción y directiva ejecutiva.

In [ ]:
import hashlib

def generate_digital_seal(order_id: str, payload_data: str, timestamp_iso: str) -> str:
    raw = f'{order_id}|{payload_data}|{timestamp_iso}'
    digest = hashlib.sha256(raw.encode('utf-8')).hexdigest()
    print(f'Sello Digital Generado para {order_id}: {digest[:16]}...{digest[-16:]}')
    return digest

seal = generate_digital_seal('ORD-DEMO-001', 'subtotal=35000;vat=7350;total=42350', '2026-11-02T06:00:00Z')
assert len(seal) == 64
print('Verificación Criptográfica: VÁLIDA ✅')

## 5. Acelerador de Carrera & Matching de Talento Tech Novus
Algoritmo de scoring técnico para evaluar candidaturas de ingeniería de datos y colocación en empresas:
- Reconstrucción de CV con estándar Silicon Valley (proyectos de streaming, Ley de Little, Welford).
- Simulación 1:1 de entrevistas de arquitectura y System Design.
- Integración directa con el ledger en partida doble (`PROD-05-CAREER-PLACEMENT`: 350 € + 21% IVA).

In [ ]:
def score_candidate_profile(role: str, experience_years: int, skills: list[str]) -> dict[str, Any]:
    base_score = min(50, experience_years * 10)
    core_skills = {'python', 'spark', 'kafka', 'sql', 'cloud', 'haskell', 'docker', 'lakehouse'}
    skill_points = sum(10 for s in skills if s.lower() in core_skills)
    total = min(100, base_score + skill_points)
    tier = 'Staff / Principal' if total >= 85 else 'Senior' if total >= 70 else 'Mid-Level'
    return {'target_role': role, 'score': total, 'tier': tier, 'placement_ready': total >= 65}

print('Evaluación de Candidato:', score_candidate_profile('Senior Data Platform Engineer', 5, ['python', 'spark', 'kafka', 'sql', 'lakehouse']))